Loading the dataset, basic information, and a brief overview

In [0]:
spark_df = spark.read.table("car_sales_cleaned")
print(f"Number of rows: {spark_df.count()}")
print(f"Number of columns: {len(spark_df.columns)}")
spark_df.show(5)

Number of rows: 2500000
Number of columns: 11
+---------+---------+-------------------+----------+------------------+-----------------+----+-----+-------+-----------+-------+
| car_make|car_model|car_production_year|sale_price|   commission_rate|commission_earned|year|month|quarter|day_of_week|car_age|
+---------+---------+-------------------+----------+------------------+-----------------+----+-----+-------+-----------+-------+
|Chevrolet|   Altima|               2017|     41875|0.0910254384830216|          3811.69|2022|    7|      3|          6|      5|
|   Nissan|Silverado|               2016|     31215|0.1341760050408948|           4188.3|2022|   12|      4|          3|      6|
|     Ford|    F-150|               2014|     49207|0.0921627911557312|          4535.05|2022|   12|      4|          2|      8|
|Chevrolet|  Corolla|               2018|     36365|0.1056897705258324|          3843.41|2023|    2|      1|          3|      5|
|    Honda|Silverado|               2020|     26248

Basic transformations performed on the dataset

In [0]:
from pyspark.sql.functions import col

# Cars older than 5 years

spark_df.filter(col("car_age") > 5).show(5)
num_of_cars_older_than_5=spark_df.filter(col("car_age") > 5).count()
print(f"The number of cars older than 5 years is {num_of_cars_older_than_5}.")

+--------+---------+-------------------+----------+------------------+-----------------+----+-----+-------+-----------+-------+
|car_make|car_model|car_production_year|sale_price|   commission_rate|commission_earned|year|month|quarter|day_of_week|car_age|
+--------+---------+-------------------+----------+------------------+-----------------+----+-----+-------+-----------+-------+
|  Nissan|    F-150|               2016|     38474|0.1344388368886588|           5172.4|2023|    3|      1|          2|      7|
|    Ford|    Civic|               2016|     33340|0.1145359215866074|          3818.63|2023|    4|      2|          5|      7|
|    Ford|   Altima|               2013|     41937|0.0921907218120205|           3866.2|2022|    9|      3|          6|      9|
|    Ford|   Altima|               2015|     14769|0.0772469351075994|          1140.86|2022|   12|      4|          6|      7|
|   Honda|    F-150|               2013|     41397|0.1427801204620234|          5910.67|2022|    6|     

In [0]:
# Average price by car make

spark_df.groupBy("car_make").agg({"sale_price":"avg"}).show()

+---------+------------------+
| car_make|   avg(sale_price)|
+---------+------------------+
|    Honda|30032.532548278665|
|   Nissan|30016.451923115466|
|   Toyota|30001.989894970877|
|     Ford| 29994.46124002313|
|Chevrolet| 30015.43155728287|
+---------+------------------+



In [0]:
# Total commission per month

spark_df.groupBy("month").agg({"commission_earned":"sum"}).sort(col("month").asc()).show()

+-----+----------------------+
|month|sum(commission_earned)|
+-----+----------------------+
|    1|   6.345018561299926E8|
|    2|   5.751764942499999E8|
|    3|   6.341318995100036E8|
|    4|   6.132741183199975E8|
|    5|   6.558508484000003E8|
|    6|   6.166333067799981E8|
|    7|   6.347542323199993E8|
|    8|   6.355677996400001E8|
|    9|   6.135596961399987E8|
|   10|   6.374891409699999E8|
|   11|   6.131065581199976E8|
|   12|   6.384673915799979E8|
+-----+----------------------+



In [0]:
from pyspark.sql.functions import col, avg

# Top 10 cars with the highest average sale price

avg_price_df = spark_df.groupBy("car_make", "car_model") \
                       .agg(avg("sale_price").alias("avg_sale_price")) \
                       .orderBy(col("avg_sale_price").desc())
avg_price_df.show(5)

+--------+---------+------------------+
|car_make|car_model|    avg_sale_price|
+--------+---------+------------------+
|  Nissan|Silverado|30075.484460877386|
|   Honda|Silverado|30068.063409305327|
|   Honda|  Corolla|30063.127903711134|
|  Toyota|    F-150|30062.873996244656|
|  Nissan|   Altima|30054.993237811985|
+--------+---------+------------------+
only showing top 5 rows


Comparison of execution time or behavior relative to Pandas

In [0]:
import time
import pandas as pd

# Spark time
start = time.time()
spark_df.groupBy("car_make").avg("sale_price").collect()
spark_vreme = time.time() - start
print(f"Spark time: {spark_vreme:.3f}s")

# Pandas time
pandas_df = spark_df.toPandas()
start = time.time()
pandas_df.groupby("car_make")["sale_price"].mean()
pandas_vreme = time.time() - start
print(f"Pandas time: {pandas_vreme:.3f}s")

Spark time: 0.571s
Pandas time: 0.110s


First, we grouped the dataset by car_make and car_model and calculated the average sale price for each combination. We then sorted the results in descending order to identify the top 5 most expensive car models. 

To compare performance with Pandas, we measured the execution time of computing the average sale price per car make. For this relatively small dataset, Pandas was faster (0.110s) than Spark (0.517s), as Pandas operates in local memory and is optimized for small datasets, while Spark introduces some overhead due to its distributed and lazy evaluation approach. However, Spark would scale much better with larger datasets.

SparkSQL query example

In [0]:
spark_df.createOrReplaceTempView("cars")

# Average price by car make
spark.sql("""
    SELECT car_make, 
           ROUND(AVG(sale_price), 2) AS avg_price,
           COUNT(*) AS total_sales
    FROM cars
    GROUP BY car_make
    ORDER BY avg_price DESC
""").show()

+---------+---------+-----------+
| car_make|avg_price|total_sales|
+---------+---------+-----------+
|    Honda| 30032.53|     500687|
|   Nissan| 30016.45|     498930|
|Chevrolet| 30015.43|     500455|
|   Toyota| 30001.99|     500147|
|     Ford| 29994.46|     499781|
+---------+---------+-----------+



In [0]:
# Average price per production year
spark.sql("""
    SELECT car_production_year,
           ROUND(AVG(sale_price), 2) AS avg_price,
           COUNT(*) AS total_sales
    FROM cars
    GROUP BY car_production_year
    ORDER BY car_production_year ASC
""").show()

+-------------------+---------+-----------+
|car_production_year|avg_price|total_sales|
+-------------------+---------+-----------+
|               2010| 30040.98|     192462|
|               2011|  29989.1|     191800|
|               2012| 30013.26|     192454|
|               2013| 29980.54|     192970|
|               2014| 30024.59|     192491|
|               2015| 30049.84|     192595|
|               2016|  30046.8|     192267|
|               2017| 30029.78|     192663|
|               2018| 29998.92|     192549|
|               2019| 30006.92|     192225|
|               2020| 30014.08|     192657|
|               2021| 29974.04|     191636|
|               2022| 29989.15|     191231|
+-------------------+---------+-----------+



In [0]:
# Number of car sales per year

spark.sql("""
    SELECT year, 
        COUNT(*) AS total_sales
    FROM cars
    GROUP BY year
    ORDER BY year ASC
""").show()

+----+-----------+
|year|total_sales|
+----+-----------+
|2022|    1674502|
|2023|     825498|
+----+-----------+

